# Analysis notebook

The first step is to narrow down the possible analytical questions to provide cleaning guidance:

The insights I will be exploring in this data pertain to the patterns in the location of permits:
1. Are there hotspots for certain types of construction?
2. Are there any temporal patterns indicating the geographical direction of growth in the city?


Import modules:

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import your helper module
import sys
sys.path.insert(0, '..')
from src.helpers import *

Load data:

In [11]:
# Load data
df = pd.read_csv('../data/building-permits.csv')

C:\Users\aylie\AppData\Local\Temp\ipykernel_15272\1494212970.py:2: DtypeWarning: Columns (23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/building-permits.csv')


### Remove Time from *issue_date*

Issue date includes time which is always set to midnight. This is not meaningful and will be removed for clarity:

In [ ]:
# Display column for demonstration of issue
df['issue_date'].head()

0    2010-02-01T00:00:00.000
1    2010-02-16T00:00:00.000
2    2010-03-19T00:00:00.000
3    2010-03-23T00:00:00.000
4    2010-04-13T00:00:00.000
Name: issue_date, dtype: object

In [13]:
# Use pd.to_datetime() to 
df['issue_date'] = pd.to_datetime(df['issue_date'])

In [15]:
# Confirm proper application
df['issue_date'].head()

0   2010-02-01
1   2010-02-16
2   2010-03-19
3   2010-03-23
4   2010-04-13
Name: issue_date, dtype: datetime64[ns]

### Remove time from *final_date*

Similar to *issue_date*, *final_date* includes messy time information that does not carry any meaning. As such, the time will be removed from this column as well:

In [16]:
# Display column for demonstration of issue
df['final_date'].head()

0    2012-01-11T00:00:00.000
1    2011-05-11T00:00:00.000
2    2010-10-27T00:00:00.000
3    2011-01-07T00:00:00.000
4    2011-07-28T00:00:00.000
Name: final_date, dtype: object

In [17]:
df['final_date'] = pd.to_datetime(df['final_date'])

In [18]:
# Confirm proper application
df['final_date'].head()

0   2012-01-11
1   2011-05-11
2   2010-10-27
3   2011-01-07
4   2011-07-28
Name: final_date, dtype: datetime64[ns]

### Assess whether filling in missing values in columns pertaining to address is worth it:

This data contains a complete *address* column, as well as *street_number*, *street_name*, *street_type*, and *street_direction* columns which store address data more atomically. The address column only has 3 missing values, whereas the more atomoic versions of it have a minimum of 272 missing values. If data in the *address* column is mostly stored in a consistent structure, it will be used to fill the missing values in other columns pertaining to address. If it is not stored in a consistent structure, geopy will be used to get coordinates instead:

In [34]:
# Use sample to get an idea of data structure
df['address'].sample(25)

113912                 662 McPhillips ST
77118                    200 Portage AVE
118798                    81 Lumsden AVE
157188    1485  Portage AVE - Unit L141B
37669                81 Garry ST Unit 54
77126                  30 Springwater RD
26487                 2610 McPhillips ST
103430                    103 Thurlby RD
93938                940 Beaverhill BLVD
36373                  29 East Plains DR
74314                       203 Thorn DR
11354                       74 Claus Bay
44868       400 Spence ST Unit 2nd level
112196                52 Berry Hill Road
113034                     99 Siskin Bay
15993                      215 Logan AVE
93578                    288 Portage AVE
102431                      43 Mira GATE
37794                    586 Waverley ST
42963                       898 Boyd AVE
19706                381 Sage Creek BLVD
4687                    774 Ingersoll ST
56515          1353 McPhillips ST Unit 3
149928         1480 Dakota Street Unit A
70265           

Observations:
- The first 'word' is always the street number
- The second word is always related to the street name
- Some street names are 2 (+?) words long, meaning the third word is not consistent
- The last word is either the street type or the unit number, with the second last word sometimes being 'UNIT' or 'Unit'
- Some rows have units that are names instead of numbers
- Some rows have dashes in the unit number
- Some of street types are all-caps abbreviations and others are the full word in title caps
- Some rows have 'Level' specified as the second last word, with the level number being the last row
- Some rows have street numbers that include a slash (e.g., 650/652 or 364 / 364)
- Some rows have 'building' and the building letter or number after the street type
- Some rows have an extra space (data is messy)

This data is not consistent at all. Coordinates will offer similar insights at less of a time cost.